In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Regressors (optional)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import joblib



In [ ]:
# Configuration
EXCEL_PATH = "nashik_dubai_shipments_with_predictions_2024.xlsx" 
SHEET_NAME = "Shipments_2024"
TEST_SIZE = 0.20
RANDOM_STATE = 42

# Training flags
TRAIN_DELAY_REG = True       # predict actual_delay_days
TRAIN_RISK_REG  = True       # predict risk_score_0_100

# Output files
CLS_OUT = Path("best_delay_classifier.joblib")
DELAY_REG_OUT = Path("best_delay_regressor.joblib")
RISK_REG_OUT = Path("best_risk_regressor.joblib")



In [ ]:
# Load Data
df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)

In [ ]:
# Time features from depart_etd (safe feature known before arrival)
df["depart_etd"] = pd.to_datetime(df["depart_etd"])
df["month"]   = df["depart_etd"].dt.month
df["quarter"] = df["depart_etd"].dt.quarter

In [ ]:
# Select a small feature set

NUMERIC = ["setpoint_c", "quantity_kg", "month", "quarter",
           "freight_usd", "insurance_usd", "inland_cost_inr"]
CATEG   = ["mode", "equipment_type", "commodity",
           "season_monsoon_IN", "season_cyclone_ARABIAN",
           "season_heat_UAE", "season_fog_UAE"]

X = df[NUMERIC + CATEG].copy()

In [ ]:
# Preprocessor (median impute+scale for nums, impute+one-hot for cats)
preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler",  StandardScaler(with_mean=False))]), NUMERIC),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("ohe",     OneHotEncoder(handle_unknown="ignore"))]), CATEG)
])

In [ ]:


# Time features from depart_etd (safe feature known before arrival)
df["depart_etd"] = pd.to_datetime(df["depart_etd"])
df["month"]   = df["depart_etd"].dt.month
df["quarter"] = df["depart_etd"].dt.quarter

# FEATURES
NUMERIC = ["setpoint_c", "quantity_kg", "month", "quarter",
           "freight_usd", "insurance_usd", "inland_cost_inr"]
CATEG   = ["mode", "equipment_type", "commodity",
           "season_monsoon_IN", "season_cyclone_ARABIAN",
           "season_heat_UAE", "season_fog_UAE"]

X = df[NUMERIC + CATEG].copy()



In [ ]:
# Preprocessor (median impute+scale for nums, impute+one-hot for cats)
preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler",  StandardScaler(with_mean=False))]), NUMERIC),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("ohe",     OneHotEncoder(handle_unknown="ignore"))]), CATEG)
])

In [ ]:
# CLASSIFICATION — Predict On-Time vs Delayed

print("\n=== Training CLASSIFICATION (on_time_flag) ===")
y_cls = df["on_time_flag"].map({"On-Time": 1, "Delayed": 0}).astype(int)

Xtr, Xte, ytr, yte = train_test_split(
    X, y_cls, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_cls
)

candidates_cls = {
    "logreg": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "rf":     RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE,
                                     class_weight="balanced_subsample"),
    "gb":     GradientBoostingClassifier(random_state=RANDOM_STATE)
}

best_name, best_acc, best_pipe = None, -np.inf, None
for name, model in candidates_cls.items():
    pipe = Pipeline([("prep", preprocess), ("clf", model)])
    pipe.fit(Xtr, ytr)
    pred = pipe.predict(Xte)
    acc = accuracy_score(yte, pred)
    print(f"{name:>6}  accuracy = {acc:.4f}")
    if acc > best_acc:
        best_name, best_acc, best_pipe = name, acc, pipe

# Report & save
pred = best_pipe.predict(Xte)
print("\nBest classifier:", best_name, f"(accuracy={best_acc:.4f})")
print("Classification report:\n", classification_report(yte, pred, target_names=["Delayed","On-Time"]))
print("Confusion matrix:\n", confusion_matrix(yte, pred))

joblib.dump(best_pipe, CLS_OUT)
print(f"Saved best classifier to: {CLS_OUT.resolve()}")




=== Training CLASSIFICATION (on_time_flag) ===
logreg  accuracy = 0.6857
    rf  accuracy = 0.7214
    gb  accuracy = 0.7000

Best classifier: rf (accuracy=0.7214)
Classification report:
               precision    recall  f1-score   support

     Delayed       0.58      0.47      0.52        45
     On-Time       0.77      0.84      0.80        95

    accuracy                           0.72       140
   macro avg       0.68      0.65      0.66       140
weighted avg       0.71      0.72      0.71       140

Confusion matrix:
 [[21 24]
 [15 80]]
Saved best classifier to: D:\shubham patil 2024\IT VEDANT\Git_hub Projects\Shipment-Predictor-main\Shipment-Predictor-main\best_delay_classifier.joblib

=== Training REGRESSION (actual_delay_days) ===
   lin  MAE = 0.800 | R2 = 0.127
    rf  MAE = 0.818 | R2 = -0.023
   gbr  MAE = 0.803 | R2 = 0.025

Best regressor (delay): lin | MAE=0.800 | R2=0.127
Saved best delay regressor to: D:\shubham patil 2024\IT VEDANT\Git_hub Projects\Shipment-Pred

In [ ]:
# REGRESSION — Predict actual_delay_days (optional)

if TRAIN_DELAY_REG and "actual_delay_days" in df.columns:
    print("\n=== Training REGRESSION (actual_delay_days) ===")
    y_reg = df["actual_delay_days"].astype(float)
    Xtr, Xte, ytr, yte = train_test_split(X, y_reg, test_size=TEST_SIZE, random_state=RANDOM_STATE)

    candidates_reg = {
        "lin": LinearRegression(),
        "rf":  RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE),
        "gbr": GradientBoostingRegressor(random_state=RANDOM_STATE)
    }

    best_name, best_mae, best_pipe = None, np.inf, None
    for name, model in candidates_reg.items():
        pipe = Pipeline([("prep", preprocess), ("reg", model)])
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte)
        mae = mean_absolute_error(yte, pred)
        r2  = r2_score(yte, pred)
        print(f"{name:>6}  MAE = {mae:.3f} | R2 = {r2:.3f}")
        if mae < best_mae:
            best_name, best_mae, best_pipe = name, mae, pipe

    pred = best_pipe.predict(Xte)
    mae = mean_absolute_error(yte, pred)
    r2  = r2_score(yte, pred)
    print(f"\nBest regressor (delay): {best_name} | MAE={mae:.3f} | R2={r2:.3f}")

    joblib.dump(best_pipe, DELAY_REG_OUT)
    print(f"Saved best delay regressor to: {DELAY_REG_OUT.resolve()}")

In [ ]:
# REGRESSION — Predict risk_score_0_100 (optional)
if TRAIN_RISK_REG and "risk_score_0_100" in df.columns:
    print("\n=== Training REGRESSION (risk_score_0_100) ===")
    y_reg = df["risk_score_0_100"].astype(float)
    Xtr, Xte, ytr, yte = train_test_split(X, y_reg, test_size=TEST_SIZE, random_state=RANDOM_STATE)

    candidates_reg = {
        "lin": LinearRegression(),
        "rf":  RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE),
        "gbr": GradientBoostingRegressor(random_state=RANDOM_STATE)
    }

    best_name, best_mae, best_pipe = None, np.inf, None
    for name, model in candidates_reg.items():
        pipe = Pipeline([("prep", preprocess), ("reg", model)])
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte)
        mae = mean_absolute_error(yte, pred)
        r2  = r2_score(yte, pred)
        print(f"{name:>6}  MAE = {mae:.3f} | R2 = {r2:.3f}")
        if mae < best_mae:
            best_name, best_mae, best_pipe = name, mae, pipe

    pred = best_pipe.predict(Xte)
    mae = mean_absolute_error(yte, pred)
    r2  = r2_score(yte, pred)
    print(f"\nBest regressor (risk): {best_name} | MAE={mae:.3f} | R2={r2:.3f}")

    joblib.dump(best_pipe, RISK_REG_OUT)
    print(f"Saved best risk regressor to: {RISK_REG_OUT.resolve()}")

print("\nDone.")



=== Training REGRESSION (risk_score_0_100) ===
   lin  MAE = 4.405 | R2 = 0.902
    rf  MAE = 3.066 | R2 = 0.950
   gbr  MAE = 3.301 | R2 = 0.943

Best regressor (risk): rf | MAE=3.066 | R2=0.950
Saved best risk regressor to: D:\shubham patil 2024\IT VEDANT\Git_hub Projects\Shipment-Predictor-main\Shipment-Predictor-main\best_risk_regressor.joblib

Done.


In [ ]:
import pandas as pd
import joblib

In [ ]:
# Load models
cls_model   = joblib.load("best_delay_classifier.joblib")   # classification (On-Time vs Delayed)
delay_model = joblib.load("best_delay_regressor.joblib")    # regression (delay days)
risk_model  = joblib.load("best_risk_regressor.joblib")     # regression (risk score)

# Sample data
sample_data = pd.DataFrame([
    {
        "setpoint_c": 4, "quantity_kg": 12000,
        "month": 7, "quarter": 3,
        "freight_usd": 1000, "insurance_usd": 5, "inland_cost_inr": 25000,
        "mode": "Sea", "equipment_type": "Reefer 40'", "commodity": "Coriander",
        "season_monsoon_IN": True, "season_cyclone_ARABIAN": False,
        "season_heat_UAE": True, "season_fog_UAE": False
    },
    {
        "setpoint_c": 7, "quantity_kg": 800,
        "month": 1, "quarter": 1,
        "freight_usd": 1500, "insurance_usd": 8, "inland_cost_inr": 18000,
        "mode": "Air", "equipment_type": "AKE ULD", "commodity": "Green Chili",
        "season_monsoon_IN": False, "season_cyclone_ARABIAN": False,
        "season_heat_UAE": False, "season_fog_UAE": True
    }
])

# Predictions
pred_cls   = cls_model.predict(sample_data)         # 1=On-Time, 0=Delayed
pred_delay = delay_model.predict(sample_data)       # days
pred_risk  = risk_model.predict(sample_data)        # 0–100

# Attach to DataFrame
sample_data["Predicted_Status"] = ["On-Time" if p==1 else "Delayed" for p in pred_cls]
sample_data["Predicted_DelayDays"] = pred_delay.round(1)
sample_data["Predicted_RiskScore"] = pred_risk.round(1)

print(sample_data[["commodity","mode","quantity_kg",
                   "Predicted_Status","Predicted_DelayDays","Predicted_RiskScore"]])



     commodity mode  quantity_kg Predicted_Status  Predicted_DelayDays  \
0    Coriander  Sea        12000          Delayed                  1.0   
1  Green Chili  Air          800          On-Time                 -4.3   

   Predicted_RiskScore  
0                 55.9  
1                 27.9  
